In [1]:
corpus = [
    "low roll fox",
    "lower"
]

def normalize_text(text):
    normal = text.strip().lower()
    return normal

def split_sentence(text):
    word_list = text.split()
    return word_list

def toCharacters(word):
    return list(word) + ['</w>']

def normalize(corpus):
    processed = []

    for sentence in corpus:
        sentence = normalize_text(sentence)
        words = split_sentence(sentence)
        sentence_list = []
        for word in words:
            sentence_list.append(toCharacters(word))

        processed.append(sentence_list)

    return processed

normalized_corpus = normalize(corpus)

print(normalized_corpus)



[[['l', 'o', 'w', '</w>'], ['r', 'o', 'l', 'l', '</w>'], ['f', 'o', 'x', '</w>']], [['l', 'o', 'w', 'e', 'r', '</w>']]]


In [2]:
def generate_initial_vocab(corpus):
    vocab = []
    for sentence in corpus:
        for word in sentence:
            for character in word:
                if character not in vocab:
                    vocab.append(character)
    return vocab


vocab = generate_initial_vocab(normalized_corpus)
print(vocab)

['l', 'o', 'w', '</w>', 'r', 'f', 'x', 'e']


In [3]:
def count_pairs(corpus):
    pairs = dict()
    word_freq = dict()
    for sentence in corpus:
        for word in sentence:
            word = tuple(word)
            if word in word_freq:
                word_freq[word] += 1
            else:
                word_freq[word] = 1
    
    for word in word_freq:
        for i in range (len(word) -1):
            pair = (word[i], word[i+1])

            
            if pair in pairs:
                pairs[pair] += word_freq[word]
            else:
                pairs[pair] = word_freq[word]
    return pairs

pairs = count_pairs(normalized_corpus)
                    
def get_frequent_pair(pairs):
    max_pair = None

    for key, value in pairs.items():
        if max_pair is None or value > pairs[max_pair]:
            max_pair = key

    return max_pair

max_pair = get_frequent_pair(pairs)

print(max_pair)


('l', 'o')


In [4]:
def merge_tokens(sentence, max_pair):
    new_sentence = []
    for word in sentence:
        new_word = []
        i = 0
        while i < (len(word)):
            if i < len(word)-1 and (word[i],word[i+1]) == max_pair:
                new_word.append(word[i] + word[i+1])
                i+=1
            else:
                new_word.append(word[i])
            i += 1
        new_sentence.append(new_word)
    return new_sentence


def merge_corpus(corpus, max_pair):
    new_corpus = []
    for sentence in corpus:
        new_sentence = merge_tokens(sentence, max_pair)
        new_corpus.append(new_sentence)

    return new_corpus
new_corpus = merge_corpus(normalized_corpus, max_pair)

print(new_corpus)

[[['lo', 'w', '</w>'], ['r', 'o', 'l', 'l', '</w>'], ['f', 'o', 'x', '</w>']], [['lo', 'w', 'e', 'r', '</w>']]]


In [5]:
corpus = [
    "low lower lowest",
    "low lower",
    "new newer newest",
    "new news",
    "wide wider widest",
    "wide widely",
    "small smaller smallest",
    "small smart",
    "smart smarter smartest",
    "fast faster fastest",
]

def bytePairEncoding(corpus, MAX_VOCAB):
    merge_rules = []

    corpus = normalize(corpus)
    vocab = generate_initial_vocab(corpus)
    while len(merge_rules) < MAX_VOCAB:
        pairs = count_pairs(corpus)
        if len(pairs) == 0: 
            break
        max_pair = get_frequent_pair(pairs)
        merge_rules.append(max_pair)
        if (max_pair[0] + max_pair[1]) not in vocab:
            vocab.append(max_pair[0] + max_pair[1])    
        corpus = merge_corpus(corpus, max_pair)
    return corpus, merge_rules, vocab

corpus, merge_rules, vocab = bytePairEncoding(corpus, 15)

print(merge_rules)
print(vocab)

[('s', 't'), ('s', 'm'), ('sm', 'a'), ('e', 'r'), ('er', '</w>'), ('st', '</w>'), ('e', 'st</w>'), ('l', 'o'), ('lo', 'w'), ('n', 'e'), ('ne', 'w'), ('w', 'i'), ('wi', 'd'), ('sma', 'l'), ('smal', 'l')]
['l', 'o', 'w', '</w>', 'e', 'r', 's', 't', 'n', 'i', 'd', 'y', 'm', 'a', 'f', 'st', 'sm', 'sma', 'er', 'er</w>', 'st</w>', 'est</w>', 'lo', 'low', 'ne', 'new', 'wi', 'wid', 'smal', 'small']


In [6]:
new_sentence = ["The cat walks quickly through the low enclosure. It feels smart as it sees new, small rabbits eat wide carrots"]

def tokenize(sentence, merge_rules):
    sentence = normalize(sentence)
    for merge in merge_rules:
        sentence = merge_tokens(sentence, merge)

    return sentence

print(tokenize(new_sentence, merge_rules))

[[['t', 'h', 'e', '</w>'], ['c', 'a', 't', '</w>'], ['w', 'a', 'l', 'k', 's', '</w>'], ['q', 'u', 'i', 'c', 'k', 'l', 'y', '</w>'], ['t', 'h', 'r', 'o', 'u', 'g', 'h', '</w>'], ['t', 'h', 'e', '</w>'], ['l', 'o', 'w', '</w>'], ['e', 'n', 'c', 'l', 'o', 's', 'u', 'r', 'e', '.', '</w>'], ['i', 't', '</w>'], ['f', 'e', 'e', 'l', 's', '</w>'], ['s', 'm', 'a', 'r', 't', '</w>'], ['a', 's', '</w>'], ['i', 't', '</w>'], ['s', 'e', 'e', 's', '</w>'], ['n', 'e', 'w', ',', '</w>'], ['s', 'm', 'a', 'l', 'l', '</w>'], ['r', 'a', 'b', 'b', 'i', 't', 's', '</w>'], ['e', 'a', 't', '</w>'], ['w', 'i', 'd', 'e', '</w>'], ['c', 'a', 'r', 'r', 'o', 't', 's', '</w>']]]


In [13]:
import pandas as pd

df = pd.read_csv("spa-eng/spa.txt", sep='\t')
df = df.drop(df.columns[2], axis = 1)
corpus = df.values.ravel().tolist()

In [11]:
corpus, merge_rules, vocab = bytePairEncoding(corpus, 10)